# 2. Quality check (find the junk responses)

This flags responses that look low-quality and recommends which to exclude. It does **not** delete anything: it writes `output/quality_report.csv` with one row per respondent, a column for each warning sign, and an `exclude_recommended` column. You decide what to do with it.

Signals (thresholds live in `output/config.json` under `quality`): **speeding** (finished implausibly fast, or clicked through pages too quickly), **low_recaptcha** (bot score below cutoff), **attention** (failed an instructed-response check, only if you defined any), **straightlining** (identical answer down a whole matrix grid).

A response is flagged for exclusion when enough independent signals trip (default 2). One stray signal is treated as noise.

In [ ]:
# --- standard setup (works on any OS; uses relative paths only) ---
from pathlib import Path
import json, re
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":          # launched from inside notebooks/
    ROOT = ROOT.parent
DATA_DIR = ROOT / "sample_data"       # put your own Qualtrics CSV here
OUT_DIR  = ROOT / "output"            # everything this tool writes lands here
OUT_DIR.mkdir(exist_ok=True)
print("Repo root :", ROOT)
print("Data dir  :", DATA_DIR)
print("Output dir:", OUT_DIR)

In [ ]:
config = json.loads((OUT_DIR / "config.json").read_text())
Q = config["quality"]
clean = pd.read_csv(OUT_DIR / "responses_clean.csv", dtype=str, keep_default_na=False)
print(f"Loaded {len(clean)} responses; "
      f"{len(config.get('straightline_groups', {}))} matrix group(s), "
      f"{len(config.get('attention_checks', {}))} attention check(s).")

## Compute the flags

Each helper returns True/False for one respondent. None of this needs editing.

In [ ]:
def num(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return None

page_cols = [c for c in clean.columns if c.endswith("_Page Submit")]
id_col = "ResponseId" if "ResponseId" in clean.columns else clean.columns[0]

def flag_speeding(row):
    dur = num(row.get("Duration (in seconds)"))
    too_short = dur is not None and dur < Q["total_duration_floor_sec"]
    fast_pages = sum(1 for c in page_cols
                     if num(row.get(c)) is not None and num(row.get(c)) < Q["page_floor_sec"])
    return bool(too_short or fast_pages >= Q["min_fast_pages_to_flag"])

def flag_recaptcha(row):
    s = num(row.get("Q_RecaptchaScore"))
    return s is not None and s < Q["recaptcha_min"]

def flag_attention(row):
    checks = config.get("attention_checks", {})
    shown = passed = failed = 0
    for code_, expected in checks.items():
        val = str(row.get(code_, "")).strip()
        if val == "":
            continue                       # blank = not shown on this branch, not a fail
        shown += 1
        if val == str(expected).strip():
            passed += 1
        else:
            failed += 1
    flagged = failed >= Q["attention_fail_min"]
    hard = shown >= 2 and passed == 0
    return flagged, hard

def flag_straightline(row):
    n_lined = 0
    for base, items in config.get("straightline_groups", {}).items():
        vals = [str(row.get(c, "")).strip() for c in items]
        vals = [v for v in vals if v != ""]
        if len(vals) >= Q["min_items_for_straightline"] and len(set(vals)) == 1:
            n_lined += 1
    return n_lined >= Q["min_straightline_matrices_to_flag"]

def flag_incomplete(row):
    fin = str(row.get("Finished", "")).strip().lower()
    prog = num(row.get("Progress"))
    return (fin in ("false", "0", "")) or (prog is not None and prog < 100)

## Build the report

In [ ]:
rows = []
for _, r in clean.iterrows():
    attn, attn_hard = flag_attention(r)
    f = {
        "speeding": flag_speeding(r),
        "low_recaptcha": flag_recaptcha(r),
        "attention": attn,
        "straightlining": flag_straightline(r),
    }
    incomplete = flag_incomplete(r)
    n_flags = sum(f.values()) + (incomplete if Q["count_incomplete_toward_exclusion"] else 0)
    exclude = bool(n_flags >= Q["exclude_min_flags"] or attn_hard)
    reasons = [k for k, v in f.items() if v] + (["incomplete"] if incomplete else [])
    rows.append({
        id_col: r.get(id_col), **f, "incomplete": incomplete,
        "num_flags": int(sum(f.values())), "exclude_recommended": exclude,
        "reasons": ", ".join(reasons),
    })

report = pd.DataFrame(rows)
report.to_csv(OUT_DIR / "quality_report.csv", index=False)
print("Wrote", OUT_DIR / "quality_report.csv")

## Summary

In [ ]:
print(f"Total responses     : {len(report)}")
for col in ["speeding", "low_recaptcha", "attention", "straightlining", "incomplete"]:
    print(f"  flagged {col:<15}: {int(report[col].sum())}")
print(f"Exclude recommended : {int(report['exclude_recommended'].sum())}")
print(f"Would keep          : {int((~report['exclude_recommended']).sum())}")
print()
report[report["exclude_recommended"]][[id_col, "num_flags", "reasons"]]